实验日期：2026年10月27日｜平台：魔搭 AMD Notebook｜投稿方向：AMD GPU激励计划 / 模型部署及推理 / AIGC

我在魔搭 AMD Notebook 上完成了 ComfyUI 和 MiniMax H3 的部署，并把日常使用的配置收敛为两个工作流：一个不使用加速 LoRA，作为质量优先的默认方案；另一个采用 Larry v4 LoRA，在更短时间内生成候选视频。此次工作重点是环境适配、配置复现和结果比较。

相同提示词、种子和分辨率下，无 LoRA 版生成一段 5.17 秒视频总耗时 649.276 秒，Larry 加速版耗时 330.593 秒。抽帧检查也暴露了代价：提示词要求狐狸站立，无 LoRA 版遵循了这一要求，而加速版生成了坐姿。因此，我保留两种选择，不把速度缩短等同于质量无损。

一、先记录真实环境，再安装依赖
---------------

这次使用的是魔搭提供的 AMD GPU 镜像。运行时可以正常调用 HIP，但设备产品名称没有正常返回，所以本文只记录可测得的架构和显存，不据此推断具体显卡型号。

项目

本次实际值

GPU 架构与显存

gfx942；196288 MiB（约192 GiB）；1张 GPU

Python / ComfyUI

3.12.13 / v0.42.5

PyTorch / HIP

2.12.0+git6bbd260 / 7.2.53211

torchvision / torchaudio

0.27.1+df56172 / 2.11.0+34c52a6

推理路径

平台 ROCm PyTorch；本轮未加入额外注意力加速后端

持久化目录 / 临时环境

/mnt/workspace / /tmp/comfyui-venv

安装前后都做了 GPU 矩阵乘法检查。ROCm 版 PyTorch 仍使用 torch.cuda 接口；判断依据是 torch.version.hip 与实际 GPU 运算，而不是接口名称。




In [ ]:
import torch
assert torch.version.hip, "当前不是可用的 ROCm PyTorch 环境"
assert torch.cuda.is_available(), "GPU 不可用"
a = torch.ones((512, 512), device="cuda")
b = a @ a
torch.cuda.synchronize()
assert b[0, 0].item() == 512
print(torch.__version__, torch.version.hip, "GPU_MATMUL_OK")



不能直接把 NVIDIA 环境的依赖安装命令整套搬过来。这里用 --system-site-packages 继承平台预装的 ROCm 组件，并固定 torch、torchvision、torchaudio 的实际版本，避免 pip 解析依赖时替换它们。




In [ ]:
torch==2.12.0+git6bbd260
torchvision==0.27.1+df56172
torchaudio==2.11.0+34c52a6



二、让安装速度和重启恢复兼顾
--------------

最初把大量 Python 小文件安装到持久化 NAS，过程很慢。最终改为在 /tmp 建立虚拟环境，安装完成后打包到 /mnt/workspace；代码、模型、工作流和视频始终放在持久化目录。这样既利用了本地磁盘安装，又保留了重启恢复所需的文件。

Git clone 曾出现 GnuTLS 连接中断，因此改用官方 GitHub 的源码归档。本次 v0.43.0 的前端依赖在可用软件源中不同步，最终使用已验证可运行的 v0.42.5；这是本次兼容性选择，不代表更新版本不可用。




In [ ]:
# 在魔搭 AMD Notebook 的 Linux 终端执行；适用于尚未安装 ComfyUI 的新目录。
cd /mnt/workspace
curl -fL --retry 3 https://codeload.github.com/Comfy-Org/ComfyUI/tar.gz/refs/tags/v0.42.5 -o ComfyUI-v0.42.5.tar.gz
tar -xzf ComfyUI-v0.42.5.tar.gz
mv ComfyUI-0.42.5 ComfyUI
mkdir -p /mnt/workspace/comfyui-install-notes

# 把附件中的 amd-constraints.txt 放进上述记录目录，再执行安装。
python -m venv --system-site-packages /tmp/comfyui-venv
/tmp/comfyui-venv/bin/python -m pip install --disable-pip-version-check --no-compile \
  --index-url https://mirrors.aliyun.com/pypi/simple/ \
  -c /mnt/workspace/comfyui-install-notes/amd-constraints.txt \
  -r /mnt/workspace/ComfyUI/requirements.txt

# 安装后重复 GPU 自检，成功后再保留运行环境备份。
tar -czf /mnt/workspace/comfyui-python-env.tar.gz -C /tmp comfyui-venv



附件保留了本次采用的安装脚本和环境版本记录。上述命令按本次成功路径整理；全新实例的镜像若改变，需要重新核对三项 ROCm 包版本，不能把约束文件当作所有镜像通用的依赖锁。

三、模型只存一份，工作流只留两套
----------------

基础权重来自 Comfy-Org/MiniMax-H3，Larry 加速权重来自作者的 MiniMax-H3-Turbo-Lora。主模型和文本编码器使用 INT8 ConvRot，视频 VAE 使用 FP16，音频 VAE 使用 FP32。本轮固定这些模型变量，没有进行 BF16 与 INT8 的画质对照。

ComfyUI 模型目录

文件

models/diffusion\_models

minimax\_h3\_fl2va\_pruned\_int8\_convrot.safetensors

models/text\_encoders

qwen3vl\_32b\_minimax\_h3\_int8\_convrot.safetensors

models/vae

minimax\_h3\_video\_vae\_fp16.safetensors

models/vae

minimax\_h3\_audio\_vae\_fp32.safetensors

models/loras

minimax\_h3\_turbo\_v4\_step600\_ema.safetensors

原始模型文件位于 /mnt/workspace 下，通过软链接接入 ComfyUI，避免复制几十 GB 的权重。ModelScope 下载采用 allow\_patterns，仅选择所需文件。下面展示最终两套工作流所需文件的下载与连接方式。




In [ ]:
from pathlib import Path
from modelscope import snapshot_download

base_files = [
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
    "vae/minimax_h3_video_vae_fp16.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
]
root = Path(snapshot_download("Comfy-Org/MiniMax-H3", revision="master",
    allow_patterns=base_files, local_dir="/mnt/workspace/MiniMax-H3"))
for relative in base_files:
    target = Path("/mnt/workspace/ComfyUI/models") / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(root / relative)

name = "minimax_h3_turbo_v4_step600_ema.safetensors"
loras = Path(snapshot_download("larryvrh/MiniMax-H3-Turbo-Lora", revision="master",
    allow_patterns=[name], local_dir="/mnt/workspace/H3-Turbo-Larry"))
target = Path("/mnt/workspace/ComfyUI/models/loras") / name
target.parent.mkdir(parents=True, exist_ok=True)
if not target.exists():
    target.symlink_to(loras / name)



下载分支可能更新，所以附件同时保留本次模型文件的 SHA256；它用于比对本次文件，不等同于上游签名。Larry 节点固定到本次实际安装的提交 4274783a23afcfdbea3b4876cb79effd6c510785。




In [ ]:
# 在自定义节点目录安装作者仓库的固定提交。
cd /mnt/workspace/ComfyUI/custom_nodes
curl -fL --retry 3 https://codeload.github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo/tar.gz/4274783a23afcfdbea3b4876cb79effd6c510785 -o /tmp/h3-larry-node.tar.gz
tar -xzf /tmp/h3-larry-node.tar.gz
mv ComfyUI-MiniMax-H3-Turbo-4274783a23afcfdbea3b4876cb79effd6c510785 ComfyUI-MiniMax-H3-Turbo
# 已有同名节点目录时，先核对版本，不要重复覆盖；安装后重启 ComfyUI。



[基础模型与官方流程](https://docs.comfy.org/tutorials/video/minimax/minimax-h3)

[Larry 作者节点和参数说明](https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo)

四、两套工作流分别承担什么任务
---------------

设置

01 无 LoRA

02 Larry v4 加速

分辨率 / 帧率

1344×768 / 24fps

1344×768 / 24fps

输入时长 / 实际输出

5秒 / 124帧，约5.17秒

5秒 / 124帧，约5.17秒

采样步数 / 调度器

20 / simple

8 / simple

采样器

res\_multistep

MiniMaxH3TurboSampler

Video / Audio shift

12 / 3

12 / 3

LoRA 加载

完全移除 LoRA 节点

MiniMaxH3TurboLoRA；v4 step600 EMA

LoRA 强度 / low\_vram

不适用

1.0 / False

最终输出设置

MP4，H.264，CRF16

MP4，H.264，CRF16

加速版采用作者推荐范围内的 8 步，配合专用 LoRA 加载器和采样器，不把任意 LoRA 与任意低步数配置混用。无加速版则使用 20 步作为质量参照。界面只保留这两个工作流，旧的 LightX2V 与预览配置移出日常工作流目录。

H3 工作流会把输入秒数对齐到模型允许的帧数。5×24=120，但按该工作流的约束向上对齐后是124帧，因此生成的视频约为5.17秒。最终两个 GUI 工作流均已打开、保存并导出执行图，核对了节点类型、模型连接和输入参数；无 LoRA 图为18个执行节点，Larry图为19个。

[工作流01 无 LoRA 20步](https://modelscope.cn/datasets/zzbbigboss/MiniMax-H3-AMD-ComfyUI-Practice/resolve/master/01_H3_Quality_NoLoRA_20step.json?view=false)

[工作流02 Larry v4 8步](https://modelscope.cn/datasets/zzbbigboss/MiniMax-H3-AMD-ComfyUI-Practice/resolve/master/02_H3_Quality_LarryV4_8step.json?view=false)

五、同一只狐狸的实测结果
------------

三次实验使用种子20261027、相同英文提示词、1344×768分辨率和124帧输出。旧LightX2V 8步样片作为已有参照。提示词同时描述主体、动作、镜头与声音：




In [ ]:
integrated_multimodal_description: [Shot 1] A red fox stands beside a shallow forest stream at sunrise. It slowly turns its head toward the camera and blinks. A gentle breeze moves its detailed fur and nearby ferns. Realistic cinematic wildlife photography, warm natural light, coherent anatomy, one continuous shot with a gentle slow push-in. No cuts, no text, no logos.

overall_soundscape: Soft flowing water, faint distant birds and gentle wind through leaves. No speech.

non_diegetic_music: N/A



方案

总耗时

抽帧中观察到的表现

无LoRA 20步

649.276秒（约10分49秒）

狐狸保持站立，头部逐渐转向镜头；更符合姿态要求

Larry v4 8步

330.593秒（约5分31秒）

毛发和面部细节清楚，转头、眨眼可见；狐狸为坐姿

旧LightX2V 8步

322.780秒（约5分23秒）

同样生成坐姿；仅作已有样片参照

这些是本机每种方案一次任务的墙钟总时间，包含模型加载与解码。任务顺序、缓存状态和输出压缩设置不同，因此它们不是严格性能基准，也不足以证明某个LoRA普遍更快或更好。本轮没有用一个比例概括所有题材的加速收益。

每段视频都完整解码得到124帧，并抽查覆盖全片的12帧。抽帧中主体和场景保持连贯，但这不等于逐帧审片、无闪烁认证或完整音轨听评。文件包含32kHz双声道AAC音轨，声音是否自然仍需试听。

无LoRA测试时保留了强度为0的旧LoRA节点，实际未启用加速；最终工作流进一步移除了这个节点。无LoRA和旧LightX2V测试片采用原默认编码，Larry片采用CRF16，最终两个工作流统一为CRF16。降低CRF只减少输出压缩损失，不能补回模型没有生成的细节，也不能据文件大小判定画质。

我的默认选择是01无LoRA版：在这一个场景中，它对“站立”的遵循更准确。02用于节约候选生成时间，但要接受效果差异。人物、大幅运动、长镜头以及参考图条件还需要独立测试。

六、云端遇到的两个接口与访问问题
----------------

第一个问题是访问入口。本次通过经典JupyterLab终端中的0.0.0.0:8188链接进入平台认证的子域名代理；旧的路径式/proxy/8188入口返回403。复现时使用自己实例提供的入口，不复制他人的实例地址。

第二个问题是工作流保存。平台网关会解码URL路径中的斜杠，导致原来的userdata路由无法匹配。针对ComfyUI v0.42.5，本次在app/user\_manager.py的现有路由注册作用域内，追加两个仅匹配顶层workflows/\*.json的别名，仍调用原读写处理函数。




In [ ]:
# 在原有 getuserdata / post_userdata 处理函数定义之后注册。
# 复用原处理逻辑的用户目录与内容校验，只兼容顶层工作流文件。
routes.get(r"/userdata/{file:workflows/[^/]+[.]json}")(getuserdata)
routes.post(r"/userdata/{file:workflows/[^/]+[.]json}")(post_userdata)



测试中，编码斜杠与解码斜杠两种读取方式都返回200，保存后内容往返校验一致；父目录越界请求仍返回403。嵌套目录未验证，所以不把这个补丁扩展成通用文件服务修复。仅在遇到相同现象、使用相同代码版本时参考附件补丁。

七、生成、下载与关闭环境
------------

使用时，在ComfyUI左侧选择01或02，修改提示词与noise\_seed，点击“运行”。完成后从视频菜单下载MP4，或在JupyterLab的ComfyUI/output/video目录中右键Download。

启动脚本放在 /mnt/workspace/comfyui-install-notes/start\_comfyui.sh。它会在实例重启后恢复临时虚拟环境，再启动服务：




In [ ]:
#!/usr/bin/env bash
# Restore the local runtime from persistent storage after an instance restart.
set -euo pipefail
if [ ! -x /tmp/comfyui-venv/bin/python ]; then
  tar -xzf /mnt/workspace/comfyui-python-env.tar.gz -C /tmp
fi
cd /mnt/workspace/ComfyUI
exec /tmp/comfyui-venv/bin/python main.py --listen 0.0.0.0 --port 8188 --disable-auto-launch

In [ ]:
bash /mnt/workspace/comfyui-install-notes/start_comfyui.sh



正常关闭实例前，应把代码、模型、工作流、视频和环境备份保存在/mnt/workspace。关闭浏览器标签不会停止GPU实例；完成生成后在魔搭页面关闭实例，下一次重新开启同类AMD环境，再运行启动脚本。本文使用的两个工作流、原片、环境记录和哈希也另存了一份本地备份。

八、复现附件与参考
---------

附件中的workflows目录只有两个GUI工作流；videos是两段实测原片；evidence包含去掉任务标识的结果汇总、模型哈希和界面执行图核对结果；reproduction保留本次使用的安装、启动与网关补丁文件。模型权重不随附件分发，从各自原仓库下载。本文没有新训练LoRA，使用的模型与自定义节点归原作者所有。

[MiniMax H3 官方工作流说明](https://docs.comfy.org/tutorials/video/minimax/minimax-h3)

[Comfy-Org MiniMax-H3 模型文件](https://modelscope.cn/models/Comfy-Org/MiniMax-H3)

[Larry v4 模型说明](https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora)

[Larry 自定义节点与示例工作流](https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo)

[LightX2V 原始参照工作流](https://github.com/ModelTC/Minimax-H3-Turbo)

[社区多场景对照页面](https://jo-nike.github.io/h3-turbo-eval/)

[魔搭 Notebook 文档](https://modelscope.cn/docs/notebooks/intro)

本次部署排障、资料核对与文稿整理使用AI编程助手协助。环境信息、耗时、文件校验与样片来自本次实际运行，社区配置来源已列明。

附：实测原片与抽帧对照
-----------

无 LoRA 20 步实测原片（约5.17秒，124帧，未重编码）：

CustomVideoPlugin

Larry v4 8 步实测原片（约5.17秒，124帧，未重编码）：

CustomVideoPlugin

无 LoRA 20 步：覆盖全片的12帧抽查，狐狸保持站立。![](https://resources.modelscope.cn/race/image/4ac342b0-67cc-46de-82cd-998b1d934505.jpeg)

Larry v4 8 步：覆盖全片的12帧抽查，细节可见，但生成了坐姿。![](https://resources.modelscope.cn/race/image/a101dde5-54a2-41c4-bb25-a21cd70168ac.jpeg)

[魔搭配套附件仓库：两个工作流、两段原片、对照图及复现记录](https://modelscope.cn/datasets/zzbbigboss/MiniMax-H3-AMD-ComfyUI-Practice/files)

[下载完整复现素材包（ZIP，含安装与启动脚本、模型哈希、环境记录）](https://modelscope.cn/datasets/zzbbigboss/MiniMax-H3-AMD-ComfyUI-Practice/resolve/master/04_%E5%85%AC%E5%BC%80%E6%8A%95%E7%A8%BF%E7%B4%A0%E6%9D%90%E5%8C%85.zip?view=false)
